# Carry + Donchian: a small editable native lab

This companion keeps one adjacent system.py and one fully resolved config.yaml.
It deliberately answers only two reusable questions:

1. How does one fixed continuous Donchian rule compare with a fixed persistent
   binary Donchian rule under identical market membership and account controls?
2. How does the carry forecast budget change the fully costed native portfolio?

Edit the constants below, restart the kernel, and run all. The default system
targets 16% annual volatility, uses a 10% forecast buffer, and never backfills
volatility. Each grid point is a fresh native System, so cached stage results
cannot leak between variants. The 2023--2026 period is already revealed and
must not be treated as a fresh holdout.

In [ ]:
%matplotlib inline
import gc
import importlib
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.utils.io import capture_output

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    candidate for candidate in (cwd, *cwd.parents)
    if (candidate / "examples/chinese_futures/research.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.chinese_futures import research as R
from sysdata.config.configdata import Config
S = importlib.import_module(
    "examples.chinese_futures.carry_donchian_lab.system"
)
R.set_notebook_style()
R.limit_blas_threads()

In [ ]:
config = Config(str(S.CONFIG_PATH))

TREND_KIND = "continuous"
CARRY_WEIGHT = 0.70
CARRY_WEIGHT_GRID = [0.00, 0.20, 0.40, 0.50, 0.60, 0.70, 0.80, 1.00]
FLAT_LAUNCH_POST_2023 = True
EVALUATION_START = pd.Timestamp("2008-07-28")
FIT_END = pd.Timestamp("2023-07-27")
AUDIT_START = pd.Timestamp("2023-07-28")
CUTOFF = pd.Timestamp("2026-07-27")
FOCUS_INSTRUMENT = "SHFE_RB"
COMPARISON_LOOKBACK = 80
TRADING_DAYS = 256.0

knobs = pd.Series({
    "trend kind": TREND_KIND,
    "chosen carry forecast weight": CARRY_WEIGHT,
    "carry-weight grid": CARRY_WEIGHT_GRID,
    "flat launch after 2023": FLAT_LAUNCH_POST_2023,
}, name="edit these constants")
display(knobs.to_frame())

## 1. One causal Chinese market panel

`CutoffChinaData` is a small `dbFuturesSimData` subclass.  It exposes only the
reviewed Chinese manifest and clips adjusted prices, multiple prices, and FX
at the requested date.  The pre-2023 system is built from a genuinely shorter
data object, not from a full-history result sliced after the fact.

Liquidity is the same rule used in the numbered series: enter at a 20-observed-
session mean of 130 contracts and exit below 70.  The decision is made at the
close; `delayfill=True` below moves the fill to the next business row.

In [ ]:
fit_data = S.CutoffChinaData(FIT_END)
audit_data = S.CutoffChinaData(CUTOFF)
FIT_INSTRUMENTS = fit_data.get_instrument_list()
AUDIT_INSTRUMENTS = audit_data.get_instrument_list()

print("reading held-contract volume once ...")
with capture_output():
    held_volume = R.held_contract_volumes(audit_data, AUDIT_INSTRUMENTS)
liquidity = R.liquidity_eligibility(
    held_volume,
    lookback=S.LIQUIDITY_LOOKBACK,
    entry_volume=S.LIQUIDITY_ENTRY,
    exit_volume=S.LIQUIDITY_EXIT,
    force_terminal_close=True,
)
fit_liquidity = liquidity.loc[:FIT_END, FIT_INSTRUMENTS]
audit_liquidity = liquidity.loc[:CUTOFF, AUDIT_INSTRUMENTS]

print(
    f"{len(FIT_INSTRUMENTS)} instruments existed by {FIT_END.date()}; "
    f"{len(AUDIT_INSTRUMENTS)} by {CUTOFF.date()}"
)

### Why a common readiness mask?

A rule that has no forecast yet must not silently change the instruments in
only one side of a comparison.  The short function below asks the native
stages for capped forecasts and volatility-sized positions, then lets every
candidate trade on the intersection.  It is experiment code, not another
framework.

In [ ]:
def common_gate(data, liquid, requests, supplied_config=None):
    supplied_config = config if supplied_config is None else supplied_config
    base_weights = R.equal_weight_panel(liquid)
    probes = [
        S.futures_system(
            data=data, config=supplied_config, eligibility=liquid,
            fixed_weights=base_weights, **request,
        )
        for request in requests
    ]
    ready = {}
    with capture_output():
        for instrument in liquid.columns:
            checks = [
                probe.combForecast.get_all_forecasts(instrument)
                .notna().all(axis=1)
                for probe in probes
            ]
            checks.append(
                probes[0].positionSize
                .get_average_position_at_subsystem_level(instrument)
                .notna()
            )
            aligned = pd.concat(checks, axis=1).reindex(liquid.index)
            ready[instrument] = aligned.ffill().fillna(False).all(axis=1)
    del probes
    gc.collect()
    return liquid & pd.DataFrame(ready, index=liquid.index)


def flat_launch(eligibility):
    launched = eligibility.copy()
    if FLAT_LAUNCH_POST_2023:
        launched.loc[launched.index < AUDIT_START] = False
    return launched

In [ ]:
def portfolio_frame(system):
    curve = system.accounts.portfolio(delayfill=True, roundpositions=True)
    frame = pd.concat({
        "gross": curve.percent.gross.as_ts,
        "costs": curve.percent.costs.as_ts,
        "net": curve.percent.as_ts,
    }, axis=1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    assert (frame["net"] - frame["gross"] - frame["costs"]).abs().max() < 1e-8
    return frame


def additive_equity(returns, starting_equity=100.0):
    returns = returns.fillna(0.0).astype(float)
    equity = starting_equity + returns.cumsum()
    anchored = pd.concat([
        pd.Series([starting_equity], index=[
            returns.index[0] - pd.Timedelta(nanoseconds=1)
        ]),
        equity,
    ])
    drawdown = anchored - anchored.cummax()
    return equity, drawdown


def performance(frame, start, end):
    returns = frame.loc[start:end, "net"].dropna().astype(float)
    annual_vol = returns.std(ddof=1) * math.sqrt(TRADING_DAYS)
    sharpe = returns.mean() / returns.std(ddof=1) * math.sqrt(TRADING_DAYS)
    arithmetic_equity, arithmetic_drawdown = additive_equity(returns)
    wealth = (1.0 + returns / 100.0).cumprod()
    anchored = pd.concat([
        pd.Series([1.0], index=[
            returns.index[0] - pd.Timedelta(nanoseconds=1)
        ]),
        wealth,
    ])
    drawdown = anchored / anchored.cummax() - 1.0
    years = (returns.index[-1] - returns.index[0]).days / 365.25
    cagr = wealth.iloc[-1] ** (1.0 / years) - 1.0
    return {
        "Sharpe": sharpe,
        "ann. vol %": annual_vol,
        "ann. arithmetic return %": returns.mean() * TRADING_DAYS,
        "total arithmetic return %": returns.sum(),
        "minimum fixed equity %": arithmetic_equity.min(),
        "additive max drawdown %": arithmetic_drawdown.min(),
        "hypothetical compound CAGR %": 100.0 * cagr,
        "hypothetical compound max drawdown %": 100.0 * drawdown.min(),
    }


def run_system(
    data, eligibility, trend_kind, carry_weight, trend_rules=None,
    supplied_config=None,
):
    supplied_config = config if supplied_config is None else supplied_config
    kwargs = {} if trend_rules is None else {"trend_rules": trend_rules}
    system = S.futures_system(
        data=data,
        config=supplied_config,
        eligibility=eligibility,
        fixed_weights=R.equal_weight_panel(eligibility),
        trend_kind=trend_kind,
        carry_weight=carry_weight,
        **kwargs,
    )
    with capture_output():
        frame = portfolio_frame(system)
    return system, frame

## 2. Continuous versus binary Donchian

This is the clean shape test.  Both sides use one fixed 80-observed-session
rule, FDM 1, source-frozen amplitude normalisation, common daily instrument
weights, the same 10% forecast buffer, and the same fully costed account.

- `breakout80` continuously measures price inside its rolling channel and
  applies the repository rule's native smoothing.
- `binary80` switches to +1 or -1 only after a strict break of the prior
  observed channel and persists until the opposite break.  Its fixed scalar
  of 10 gives a mature absolute capped forecast of 10.

In [ ]:
lookback = COMPARISON_LOOKBACK
comparison_rules = {
    "continuous": [f"breakout{lookback}"],
    "binary": [f"binary{lookback}"],
}
pair_requests = [
    dict(carry_weight=0.0, trend_kind=kind, trend_rules=rules)
    for kind, rules in comparison_rules.items()
]
fit_pair_gate = common_gate(fit_data, fit_liquidity, pair_requests)
audit_pair_gate = flat_launch(
    common_gate(audit_data, audit_liquidity, pair_requests)
)

pair_systems, pair_frames, rows = {}, {}, []
for period, data, gate, start, end in [
    ("pre-2023", fit_data, fit_pair_gate, EVALUATION_START, FIT_END),
    ("revealed post-2023", audit_data, audit_pair_gate, AUDIT_START, CUTOFF),
]:
    for kind, rules in comparison_rules.items():
        print(f"running {period}: {kind}{lookback} ...")
        system, frame = run_system(data, gate, kind, 0.0, rules)
        pair_systems[(period, kind)] = system
        pair_frames[(period, kind)] = frame
        rows.append({
            "period": period,
            "rule": kind,
            **performance(frame, start, end),
        })

donchian_comparison = pd.DataFrame(rows).set_index(["period", "rule"])
display(donchian_comparison)

In [ ]:
def pooled_forecast_scale(system, rule, eligibility, end):
    observations = []
    with capture_output():
        for instrument in eligibility.columns:
            forecast = system.forecastScaleCap.get_capped_forecast(
                instrument, rule
            ).reindex(eligibility.index)
            observations.append(
                forecast.where(eligibility[instrument]).loc[:end].dropna()
            )
    pooled = pd.concat(observations)
    return {
        "mean abs forecast": pooled.abs().mean(),
        "cap rate": pooled.abs().ge(20.0 - 1e-10).mean(),
        "observations": len(pooled),
    }


scale_check = pd.DataFrame({
    kind: pooled_forecast_scale(
        pair_systems[("pre-2023", kind)], rules[0], fit_pair_gate, FIT_END
    )
    for kind, rules in comparison_rules.items()
}).T
display(scale_check)
amplitude_matched_binary_scalar = (
    config.forecast_scalars[f"binary{lookback}"]
    * scale_check.loc["continuous", "mean abs forecast"]
    / scale_check.loc["binary", "mean abs forecast"]
)
print(
    "The source-frozen scalars are not an exact China amplitude match. "
    f"Using pre-2023 forecasts only, an amplitude-matched binary scalar "
    f"would be {amplitude_matched_binary_scalar:.3f}; it is shown as a "
    "diagnostic and is not substituted after seeing returns."
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, period, start, end in [
    (axes[0], "pre-2023", EVALUATION_START, FIT_END),
    (axes[1], "revealed post-2023", AUDIT_START, CUTOFF),
]:
    curves = pd.concat({
        kind: pair_frames[(period, kind)].loc[start:end, "net"]
        for kind in comparison_rules
    }, axis=1).fillna(0.0)
    R.cumulative_from_zero(curves).plot(
        ax=ax, title=f"{period}: cumulative net return %"
    )
plt.tight_layout()

## 3. Carry + continuous Donchian weight sweep

For the intended system, carry10/30/60/125 share the carry sleeve equally and
breakout40/80/160 share the trend sleeve equally.  Thus a 70% carry setting is
`70% / 4` per carry rule and `30% / 3` per continuous rule.  The factory omits
zero-weight rules at the 0% and 100% endpoints.

All weights use FDM 1 and the same readiness/membership panel.  This isolates
the forecast ratio: no ratio is helped by a separately fitted diversification
multiplier.  If flat launch is enabled, each post-2023 candidate starts with
zero positions, waits for the normal delayed first fill, and pays entry costs.

In [ ]:
trend_kind = TREND_KIND
grid_request = [dict(carry_weight=0.5, trend_kind=trend_kind)]
fit_grid_gate = common_gate(fit_data, fit_liquidity, grid_request)
audit_grid_gate = flat_launch(
    common_gate(audit_data, audit_liquidity, grid_request)
)

chosen_weight = float(CARRY_WEIGHT)
carry_weights = sorted(
    set(float(value) for value in CARRY_WEIGHT_GRID)
    | {chosen_weight}
)
ratio_frames, ratio_rows = {}, []
chosen_systems = {}

for carry_weight in carry_weights:
    print(f"running {carry_weight:.0%} carry / {1-carry_weight:.0%} {trend_kind} ...")
    fit_system, fit_frame = run_system(
        fit_data, fit_grid_gate, trend_kind, carry_weight
    )
    post_system, post_frame = run_system(
        audit_data, audit_grid_gate, trend_kind, carry_weight
    )
    ratio_frames[("pre-2023", carry_weight)] = fit_frame
    ratio_frames[("revealed post-2023", carry_weight)] = post_frame
    fit_stats = performance(fit_frame, EVALUATION_START, FIT_END)
    post_stats = performance(post_frame, AUDIT_START, CUTOFF)
    ratio_rows.append({
        "carry forecast weight": carry_weight,
        **{f"pre {key}": value for key, value in fit_stats.items()},
        **{f"post {key}": value for key, value in post_stats.items()},
    })
    if carry_weight == chosen_weight:
        chosen_systems = {"pre": fit_system, "post": post_system}
    else:
        del fit_system, post_system
        gc.collect()

ratio_table = pd.DataFrame(ratio_rows).set_index("carry forecast weight")
display(ratio_table)

In [ ]:
ax = ratio_table[["pre Sharpe", "post Sharpe"]].plot(
    marker="o", figsize=(9, 4),
    title=f"carry / {trend_kind} forecast budget",
)
ax.axvline(chosen_weight, color="black", linestyle="--", alpha=0.5)
ax.set_xlabel("carry forecast weight")
ax.set_ylabel("net Sharpe")
plt.tight_layout()

best_pre = ratio_table["pre Sharpe"].idxmax()
print(
    f"Highest displayed pre-2023 Sharpe: {best_pre:.0%} carry. "
    "The revealed post-2023 line is descriptive and cannot vote."
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, period, start, end in [
    (axes[0], "pre-2023", EVALUATION_START, FIT_END),
    (axes[1], "revealed post-2023", AUDIT_START, CUTOFF),
]:
    net = ratio_frames[(period, chosen_weight)].loc[start:end, "net"]
    R.cumulative_from_zero(net).plot(
        ax=ax,
        title=f"configured {chosen_weight:.0%} carry: {period}",
    )
plt.tight_layout()

## The native API, by hand

These are the useful calls when you want to inspect the result yourself.
Work from forecasts downstream to positions and finally the costed portfolio:

```python
system.combForecast.get_forecast_weights(instrument)
system.combForecast.get_all_forecasts(instrument)
system.combForecast.get_combined_forecast(instrument)
system.positionSize.get_subsystem_position(instrument)
system.portfolio.get_notional_position(instrument)
system.accounts.get_buffered_position(instrument, roundpositions=True)
system.accounts.portfolio(delayfill=True, roundpositions=True)
system.accounts.portfolio(...).percent.to_frame()
system.rawdata.get_daily_vol_normalised_returns(instrument)
```

The first object is the signal budget.  The realised volatility shown above
comes only after forecasts, volatility sizing, changing instrument membership,
IDM, buffering, integer contracts, and costs have all acted.

In [ ]:
system = chosen_systems["post"]
instrument = FOCUS_INSTRUMENT

print("exact fixed forecast weights:")
display(pd.Series(system.config.forecast_weights, name="weight").to_frame())

with capture_output():
    forecast_weights = system.combForecast.get_forecast_weights(instrument)
    forecasts = system.combForecast.get_all_forecasts(instrument)
    combined = system.combForecast.get_combined_forecast(instrument)
    subsystem = system.positionSize.get_subsystem_position(instrument)
    notional = system.portfolio.get_notional_position(instrument)
    buffered = system.accounts.get_buffered_position(
        instrument, roundpositions=True
    )

display(pd.concat({
    "combined forecast": combined,
    "subsystem position": subsystem,
    "notional position": notional,
    "buffered position": buffered,
}, axis=1).dropna(how="all").tail(10))
display(forecast_weights.dropna(how="all").tail(3))
display(forecasts.dropna(how="all").tail(3))

chosen_post = ratio_frames[("revealed post-2023", chosen_weight)]
if FLAT_LAUNCH_POST_2023:
    before = buffered.loc[buffered.index < AUDIT_START].fillna(0.0)
    assert before.eq(0.0).all()
    pre_cost = chosen_post.loc[
        chosen_post.index < AUDIT_START, "costs"
    ].abs().max()
    assert not np.isfinite(pre_cost) or pre_cost < 1e-12
    first_cost = chosen_post.loc[AUDIT_START:, "costs"]
    first_cost = first_cost[first_cost.abs() > 1e-12].head(1)
    assert len(first_cost) == 1 and first_cost.iloc[0] < 0.0
    print("flat before launch: yes")
    print("first delayed entry cost:", first_cost.to_dict())
else:
    print("post-2023 mode: ongoing positions (not a flat launch)")

In [ ]:
pre_binary = donchian_comparison.loc[("pre-2023", "binary"), "Sharpe"]
pre_continuous = donchian_comparison.loc[
    ("pre-2023", "continuous"), "Sharpe"
]
post_binary = donchian_comparison.loc[
    ("revealed post-2023", "binary"), "Sharpe"
]
post_continuous = donchian_comparison.loc[
    ("revealed post-2023", "continuous"), "Sharpe"
]

print("DATA-DRIVEN SUMMARY")
print(
    f"Fixed {lookback}-day trend Sharpe, pre-2023: "
    f"continuous {pre_continuous:.3f}, binary {pre_binary:.3f}."
)
print(
    f"Revealed post-2023: continuous {post_continuous:.3f}, "
    f"binary {post_binary:.3f}."
)
print(
    f"Configured carry/{trend_kind} mix: {chosen_weight:.0%}/"
    f"{1-chosen_weight:.0%}; pre Sharpe "
    f"{ratio_table.loc[chosen_weight, 'pre Sharpe']:.3f}, revealed post Sharpe "
    f"{ratio_table.loc[chosen_weight, 'post Sharpe']:.3f}."
)
print(
    "Controls: 16% volatility target, 10% forecast buffer, "
    "no volatility backfill, native delayed whole-contract accounts."
)